# 1. Extracción y procesamiento de datos

## 1.1. Introducción

Este notebook constituye la **fase inicial del proyecto “Análisis Multimodal de Snacks Saludables”**, cuyo propósito es ofrecer a la empresa una visión profunda del mercado de snacks saludables a partir de **datos reales extraídos de múltiples fuentes**.  

El objetivo central de esta etapa es **recopilar, unificar y preprocesar información textual** proveniente de diversas fuentes, como artículos especializados, blogs, medios de comunicación y transcripciones de podcasts, que tratan sobre hábitos de consumo, tendencias de mercado, productos y atributos valorados en snacks saludables.  

El enfoque multimodal permite combinar distintas fuentes de información y distintos formatos (texto web, audio de YouTube, podcasts) para construir un **dataset robusto y limpio**, listo para su análisis mediante modelos de lenguaje generativo en Notebook 2.  

Durante este notebook se emplean **herramientas avanzadas de web scraping, extracción de contenido y procesamiento de audio**, así como técnicas de limpieza y normalización de datos. Se incluyen pasos que garantizan:
- **Integridad de los datos:** eliminación de duplicados y contenido irrelevante.
- **Uniformidad de los formatos:** estandarización de texto y estructura de columnas.
- **Trazabilidad y reproducibilidad:** registro de fuentes y procedimientos.

Este enfoque asegura que el análisis posterior sea **fiable, estructurado y alineado con los objetivos estratégicos del proyecto**, permitiendo a la empresa tomar decisiones fundamentadas sobre innovación de productos, tendencias de mercado y preferencias de los consumidores.

**Objetivos**

1. **Recolectar datos de múltiples fuentes** sobre snacks saludables:
   - Artículos de medios especializados y blogs.
   - Podcasts y transcripciones de YouTube.
   - Otros contenidos relevantes de Internet (RSS, páginas web).  

2. **Extraer y normalizar el contenido textual**:
   - Limpieza de HTML y scripts.
   - Extracción de texto principal.
   - Procesamiento de transcripciones de audio a texto.

3. **Crear un dataset multimodal unificado**, integrando información de diferentes fuentes en un formato homogéneo:
   - Columnas para texto limpio (`content_clean`), títulos, URLs, fechas y tipo de fuente.
   - Garantizar que los datos estén listos para análisis de LLM en Notebook 2.

4. **Documentar las fuentes y características de los datos**:
   - Clasificación por tipo: blogs, artículos, podcasts.
   - Registro de URL, fecha de publicación y otros metadatos relevantes.
   - Esta documentación permite **justificar el origen de los insights** y mantener transparencia metodológica.

5. **Preparar el dataset para análisis avanzado**:
   - Garantizar que los datos estén limpios, completos y estructurados.
   - Asegurar que columnas clave como `content_clean` y `Productos` estén listas para alimentar el LLM en la siguiente fase.

6. **Establecer una base reproducible**:
   - Crear scripts y procedimientos que permitan actualizar o ampliar el dataset en el futuro.
   - Facilitar la **automatización de la recolección y limpieza** de nuevas fuentes sin comprometer la calidad de los datos.

## 1.2. Librerías y rutas

En esta sección se importan las librerías necesarias y se definen las rutas de los directorios de datos y de los ejecutables que se usarán durante el proyecto.

In [ ]:
# ==============================
# LIBRERÍAS
# ==============================
import os                 # Manejo de rutas y sistema de archivos
import re                 # Expresiones regulares para limpieza de texto
import csv                # Lectura y escritura de archivos CSV
from pathlib import Path  # Manejo de rutas de forma más segura y multiplataforma
import pandas as pd       # Manejo de dataframes y análisis de datos
import requests           # Descarga de contenido web
import feedparser         # Lectura de feeds RSS
from bs4 import BeautifulSoup  # Parseo de HTML para scraping
from newspaper import Article   # Extracción de artículos web
from urllib.parse import urljoin  # Construcción de URLs absolutas
from yt_dlp import YoutubeDL     # Descarga de vídeos/audio de YouTube
import whisper            # Transcripción de audio a texto con modelo Whisper
import subprocess         # Ejecución de comandos del sistema (ffmpeg)

In [ ]:
# ==============================
# PATHS BASE
# ==============================
PROJECT_PATH = Path(r"h:\Cristina\Mis documentos\Github\TFM-2-A.Multimodal_snacks_saludables")

# Carpetas de datos
WEB_PATH = PROJECT_PATH / "data/raw/web_scraping"
AUDIO_PATH = PROJECT_PATH / "data/raw/audio"
TRANSCRIPTS_PATH = PROJECT_PATH / "data/raw/audio/transcripts"   # Carpeta para transcripciones intermedias
OUTPUT_PATH = PROJECT_PATH / "data/processed"

# Crear carpetas si no existen
for p in [WEB_PATH, AUDIO_PATH, TRANSCRIPTS_PATH, OUTPUT_PATH]:
    p.mkdir(parents=True, exist_ok=True)

# Rutas a ejecutables ffmpeg y ffprobe (ahora en la misma carpeta que el notebook)
FFMPEG_EXE = PROJECT_PATH / "ffmpeg.exe"
FFPROBE_EXE = PROJECT_PATH / "ffprobe.exe"

print(f"✅ Carpetas listas:\n- Web: {WEB_PATH}\n- Audio: {AUDIO_PATH}\n- Transcripts: {TRANSCRIPTS_PATH}\n- Output: {OUTPUT_PATH}")
print(f"✅ Ejecutables:\n- ffmpeg: {FFMPEG_EXE}\n- ffprobe: {FFPROBE_EXE}")

✅ Carpetas listas:
- Web: h:\Cristina\Mis documentos\Github\TFM-2-A.Multimodal_snacks_saludables\data\raw\web_scraping
- Audio: h:\Cristina\Mis documentos\Github\TFM-2-A.Multimodal_snacks_saludables\data\raw\audio\youtube
- Transcripts: h:\Cristina\Mis documentos\Github\TFM-2-A.Multimodal_snacks_saludables\data\raw\audio\transcripts
- Output: h:\Cristina\Mis documentos\Github\TFM-2-A.Multimodal_snacks_saludables\data\processed
✅ Ejecutables:
- ffmpeg: h:\Cristina\Mis documentos\Github\TFM-2-A.Multimodal_snacks_saludables\ffmpeg.exe
- ffprobe: h:\Cristina\Mis documentos\Github\TFM-2-A.Multimodal_snacks_saludables\ffprobe.exe


## 1.3. Extracción de datos

Se definen las fuentes de información (RSS, webs, vídeos de YouTube) y las palabras clave que se usarán para filtrar contenidos relevantes sobre snacks saludables.

En esta sección definimos las **fuentes de información** y las **palabras clave** que utilizaremos para recopilar datos sobre snacks saludables. Se han seleccionado varias fuentes de distintos tipos para asegurar un dataset multimodal que capture distintas perspectivas del mercado:

| Fuente | Tipo | Descripción | Justificación |
|--------|------|-------------|---------------|
| vitonica | Blog / Artículos | Blog de nutrición y fitness con contenidos sobre alimentación saludable. | Permite recoger artículos y consejos prácticos sobre snacks y tendencias saludables. |
| nutt | Blog / Artículos | Blog especializado en nutrición y productos saludables. | Contenido específico sobre snacks y barritas energéticas, útil para detectar productos y preferencias. |
| solnatural | Blog / Artículos | Blog de empresa de productos naturales. | Ofrece información sobre productos comerciales y nuevas tendencias en snacks. |
| centronutricion | Blog / Artículos | Blog de centro de nutrición profesional. | Contenido con enfoque nutricional y recomendaciones sobre alimentos saludables. |
| inforetail | Revista online | Noticias de alimentación y retail. | Información sobre el mercado, lanzamientos y tendencias de consumo de snacks. |
| directoalpaladar | Blog / Artículos | Blog de recetas saludables. | Contenido práctico sobre cómo incorporar snacks saludables en la dieta. |
| YouTube (varios) | Vídeos / Podcasts | Canales especializados en nutrición, ideas de snacks y entrevistas a expertos. | Permite extraer opiniones de consumidores, recomendaciones y análisis de productos de manera audiovisual. |

In [ ]:
# ==============================
# FUENTES
# ==============================
RSS_SOURCES = {
    "vitonica": ["https://www.vitonica.com/tag/snacks/rss2.xml"],
    "nutt": ["https://www.nutt.es/blog/"],
    "solnatural": ["https://solnatural.bio/blog"],
    "centronutricion":["https://www.centrojuliafarre.es/blog/"],
    "inforetail":["https://www.revistainforetail.com/noticias/alimentaci%C3%B3n"],
    "directoalpaladar":["https://www.directoalpaladar.com/tag/recetas-saludables"]
}
VIDEOS_DICT = {
    "ideas_snacks": "https://www.youtube.com/watch?v=ZPWQ2Yl8gdY",
    "podcast_recomendaciones": "https://www.youtube.com/watch?v=f3SEfE_52WY",
    "empresa_snacks": "https://www.youtube.com/watch?v=0T1yz7hMiYk",
    "realfooding_snacks": "https://www.youtube.com/watch?v=dhQ9DvzMVfY",
    "no_experto_snacks": "https://www.youtube.com/watch?v=0T1yz7hMiYk"
}

Se definen palabras clave relacionadas con snacks saludables, ingredientes, atributos nutricionales y formas de consumo (por ejemplo: “snack saludable”, “barrita energética”, “frutos secos”, “alto en fibra”, “sin gluten”, “merienda”, “picoteo”).  
Estas keywords permiten **filtrar los artículos y transcripciones** para centrarnos únicamente en información relevante para el estudio de mercado.  

In [ ]:
# ==============================
# KEYWORDS
# ==============================
KEYWORDS = [
    "snack", "healthy", "saludable", "barrita", "chips", "fruta", "sabor", "precio",
    "frutos secos", "nueces", "almendras", "pistachos", "semillas", "granola", "crackers",
    "palitos de pan", "tortitas de arroz", "avena", "proteína", "fibra", "cacao",
    "mantequilla de almendra", "chia", "linaza", "muesli", "yogur", "pasas", "coco rallado",
    "bajo en azúcar", "sin azúcar", "bajo en grasa", "sin gluten", "orgánico", "energético",
    "ligero", "rico en proteínas", "alto en fibra", "merienda", "tentempié", "desayuno",
    "media mañana", "picoteo", "chips vegetales", "galleta saludable"
]

KEYWORD_PAT = re.compile("|".join([re.escape(k) for k in KEYWORDS]), re.IGNORECASE)

### A) Scraping web

Para los blogs y artículos, usamos un **scraping combinado de RSS y HTML**: primero obtenemos los enlaces desde los feeds RSS, y si no están disponibles, parseamos la web con `BeautifulSoup` para extraer URLs relevantes.  

In [ ]:
# ==============================
# FUNCIONES SCRAPING WEB
# ==============================
def get_urls(url_list, domain_key):
    urls = []
    for source in url_list:
        if "rss" in source or "xml" in source:
            try:
                feed = feedparser.parse(source)
                for entry in feed.entries:
                    urls.append(entry.link)
            except: pass
        else:
            try:
                r = requests.get(source, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
                soup = BeautifulSoup(r.content, "html.parser")
                for a in soup.find_all("a", href=True):
                    full = urljoin(source, a['href'])
                    if domain_key in full: urls.append(full)
            except: pass
    return list(set(urls))[:8]

def run_web_scraping():
    print("\n🚀 INICIANDO SCRAPING WEB...")
    for source_name, links in RSS_SOURCES.items():
        site_path = WEB_PATH / source_name
        site_path.mkdir(exist_ok=True)
        urls = get_urls(links, source_name)
        count = 0
        for url in urls:
            try:
                art = Article(url)
                art.download()
                art.parse()
                if not KEYWORD_PAT.search(art.text): continue
                safe_title = re.sub(r'[^\w\s-]', '', art.title.lower().strip()).replace(' ', '_')[:50]
                file_path = site_path / f"{safe_title}.txt"
                if not file_path.exists():
                    file_path.write_text(f"TITULO: {art.title}\nURL: {url}\n{'-'*20}\n{art.text}", encoding="utf-8")
                    count += 1
            except: continue
        print(f"   ✅ {source_name}: {count} artículos nuevos.")

In [ ]:
# ==============================
# CARGA ARTÍCULOS WEB
# ==============================
def load_web_articles():
    data = []
    for site in WEB_PATH.iterdir():
        if site.is_dir():
            for file in site.rglob("*.txt"):
                try:
                    content = file.read_text(encoding="utf-8", errors="ignore")
                    content = " ".join(content.split())
                    data.append({
                        "source": site.name,
                        "title": file.stem,
                        "type": "web",
                        "content": content
                    })
                except: pass
    print(f"🌐 Artículos web cargados: {len(data)}")
    return pd.DataFrame(data)

In [ ]:
# ==============================
# EJECUCIÓN FUNCIONES
# ==============================
run_web_scraping()
df_web = load_web_articles()


🚀 INICIANDO SCRAPING WEB...
   ✅ vitonica: 8 artículos nuevos.
   ✅ nutt: 2 artículos nuevos.
   ✅ solnatural: 1 artículos nuevos.
   ✅ centronutricion: 0 artículos nuevos.
   ✅ inforetail: 0 artículos nuevos.
   ✅ directoalpaladar: 1 artículos nuevos.
🌐 Artículos web cargados: 12


In [ ]:
df_web

,source,title,type,content
0,directoalpaladar,recetas_saludables_contra_el_frío_invernal_en_...,web,TITULO: Recetas saludables contra el frío inve...
1,nutt,ejercicio_físico_en_verano,web,TITULO: Ejercicio físico en verano URL: https:...
2,nutt,el_agua_en_las_comidas_no_engorda,web,TITULO: El agua en las comidas no engorda URL:...
3,solnatural,productos_bio_veganos_sin_gluten_sin_azúcar,web,"TITULO: Productos bio, veganos, sin gluten, si..."
4,vitonica,calabacín_rebozado_y_crujiente_en_freidora_de_...,web,TITULO: Calabacín rebozado y crujiente en frei...
5,vitonica,el_mayor_experto_en_longevidad_del_mundo_revel...,web,TITULO: El mayor experto en longevidad del mun...
6,vitonica,el_picoteo_puede_arruinar_los_beneficios_de_co...,web,TITULO: El picoteo puede arruinar los benefici...
7,vitonica,los_tres_mejores_snacks_ricos_en_proteínas_y_b...,web,TITULO: Los tres mejores snacks ricos en prote...
8,vitonica,marcos_vázquez_creador_de_fitness_revolucionar...,web,"TITULO: Marcos Vázquez, creador de Fitness Rev..."
9,vitonica,nueces_crujientes_a_las_finas_hierbas_receta_s...,web,TITULO: Nueces crujientes a las finas hierbas:...


#### Filtrado de webs

Se aplican filtros para mantener solo los artículos relevantes y eliminar contenido irrelevante, asegurando un dataset de calidad.
Se muestran estadísticas de filas originales vs filas tras limpieza.

In [ ]:
# ==============================
# LIMPIEZA FLEXIBLE DE ARTÍCULOS WEB
# ==============================
def clean_web_dataframe_flexible(df):
    # Keywords relevantes: ampliadas para incluir tu lista original y términos comunes
    RELEVANT_KEYWORDS = [
        "snack", "snack saludable", "barrita", "barrita energética",
        "chips", "chips de fruta", "frutos secos", "snacks saludables",
        "fruta", "sabor", "precio", "healthy"
    ]

    # Keywords negativas más estrictas: solo eliminar artículos claramente irrelevantes
    NEGATIVE_KEYWORDS = [
        "roscon", "navidad", "galleta de navidad", "reyes", "fiesta", "pastel"
    ]

    # Crear regex para búsqueda
    RELEVANT_PAT = re.compile("|".join([re.escape(k) for k in RELEVANT_KEYWORDS]), re.IGNORECASE)
    NEGATIVE_PAT = re.compile("|".join([re.escape(k) for k in NEGATIVE_KEYWORDS]), re.IGNORECASE)

    # Filtrar artículos relevantes
    mask_relevant = df['content'].str.contains(RELEVANT_PAT) | df['title'].str.contains(RELEVANT_PAT)

    # Filtrar negativos estrictos solo en títulos
    mask_negative = df['title'].str.contains(NEGATIVE_PAT)

    # Mantener solo filas relevantes y que no tengan negativas en el título
    df_clean = df[mask_relevant & ~mask_negative].copy()

    print(f"Filas originales: {len(df)}, Filas tras limpieza flexible: {len(df_clean)}")
    return df_clean

# ==============================
# APLICAR LIMPIEZA FLEXIBLE
# ==============================
df_web_clean = clean_web_dataframe_flexible(df_web)


Filas originales: 12, Filas tras limpieza flexible: 10


In [ ]:
df_web_clean

,source,title,type,content
0,directoalpaladar,recetas_saludables_contra_el_frío_invernal_en_...,web,TITULO: Recetas saludables contra el frío inve...
1,nutt,ejercicio_físico_en_verano,web,TITULO: Ejercicio físico en verano URL: https:...
4,vitonica,calabacín_rebozado_y_crujiente_en_freidora_de_...,web,TITULO: Calabacín rebozado y crujiente en frei...
5,vitonica,el_mayor_experto_en_longevidad_del_mundo_revel...,web,TITULO: El mayor experto en longevidad del mun...
6,vitonica,el_picoteo_puede_arruinar_los_beneficios_de_co...,web,TITULO: El picoteo puede arruinar los benefici...
7,vitonica,los_tres_mejores_snacks_ricos_en_proteínas_y_b...,web,TITULO: Los tres mejores snacks ricos en prote...
8,vitonica,marcos_vázquez_creador_de_fitness_revolucionar...,web,"TITULO: Marcos Vázquez, creador de Fitness Rev..."
9,vitonica,nueces_crujientes_a_las_finas_hierbas_receta_s...,web,TITULO: Nueces crujientes a las finas hierbas:...
10,vitonica,siete_alimentos_saludables_que_están_listos_pa...,web,TITULO: Siete alimentos saludables que están l...
11,vitonica,tres_snacks_saludables_de_mercadona_que_no_pue...,web,TITULO: Tres snacks saludables de Mercadona qu...


### B) Transcripción audios Youtube

Para los vídeos y podcasts de YouTube, se descarga el audio con `yt_dlp` y se transcribe con `Whisper` para convertir el contenido audiovisual en texto, integrándolo así en el análisis multimodal.  

In [ ]:
# ==============================
# DESCARGA DE AUDIO YOUTUBE
# ==============================
def run_audio_download():
    print("\n🚀 INICIANDO DESCARGA DE AUDIO...")
    ydl_opts = {
        "format": "bestaudio/best",
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "mp3", "preferredquality": "192"}],
        "noplaylist": True,
        "quiet": True,
        "no_warnings": True,
        "ffmpeg_location": str(FFMPEG_EXE)
    }

    for nombre, url in VIDEOS_DICT.items():
        target_file = AUDIO_PATH / f"{nombre}.mp3"
        if target_file.exists() and target_file.stat().st_size > 1000:
            print(f"   ℹ️ {nombre}.mp3 ya existe y es válido.")
            continue
        print(f"   🎥 Descargando: {nombre}...")
        current_opts = ydl_opts.copy()
        current_opts["outtmpl"] = str(AUDIO_PATH / f"{nombre}.%(ext)s")
        try:
            with YoutubeDL(current_opts) as ydl:
                ydl.download([url])
            print(f"      ✅ Descarga OK.")
        except Exception as e:
            print(f"      ❌ Error descargando: {e}")

In [ ]:
# ==============================
# CARPETA DE TRANSCRIPCIONES INTERMEDIAS
# ==============================
TRANSCRIPTS_PATH = AUDIO_PATH / "transcripts"
TRANSCRIPTS_PATH.mkdir(exist_ok=True)

# ==============================
# 4. MÓDULO DE DESCARGA DE AUDIO
# ==============================
def run_audio_download():
    print("\n🚀 INICIANDO DESCARGA DE AUDIO...")
    
    ydl_opts = {
        "format": "bestaudio/best",
        "postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "mp3", "preferredquality": "192"}],
        "ffmpeg_location": str(PROJECT_PATH),  # Carpeta donde está ffmpeg.exe
        "noplaylist": True,
        "quiet": True,
        "no_warnings": True
    }

    for nombre, url in VIDEOS_DICT.items():
        target_file = AUDIO_PATH / f"{nombre}.mp3"
        
        if target_file.exists() and target_file.stat().st_size > 1000:
            print(f"   ℹ️ {nombre}.mp3 ya existe y es válido.")
            continue
            
        print(f"   🎥 Descargando: {nombre}...")
        current_opts = ydl_opts.copy()
        current_opts["outtmpl"] = str(AUDIO_PATH / f"{nombre}.%(ext)s")
        
        try:
            with YoutubeDL(current_opts) as ydl:
                ydl.download([url])
            print(f"      ✅ Descarga OK.")
        except Exception as e:
            print(f"      ❌ Error descargando: {e}")

# ==============================
# 5. TRANSCRIPCIÓN YOUTUBE + GUARDAR TXT
# ==============================
def transcribe_youtube():
    print("\n🧠 Cargando modelo Whisper (base)...")
    model = whisper.load_model("base")
    data = []
    mp3_files = list(AUDIO_PATH.glob("*.mp3"))

    if not mp3_files:
        print("⚠️ No hay archivos mp3 para transcribir.")
        return pd.DataFrame()

    for file in mp3_files:
        print(f"🎧 Transcribiendo {file.name}")
        try:
            result = model.transcribe(str(file), language="es", fp16=False)
            text_content = " ".join(result["text"].split())

            # Guardar transcripción en .txt
            transcript_file = TRANSCRIPTS_PATH / f"{file.stem}.txt"
            transcript_file.write_text(
                f"TITULO: {file.stem}\nSOURCE: Youtube\n{'-'*20}\n{text_content}",
                encoding="utf-8"
            )

            # Agregar al DataFrame
            data.append({
                "source": "youtube",
                "title": file.stem,
                "type": "audio",
                "content": text_content
            })

            print(f"   ✅ Transcripción completa (guardada en {transcript_file.name})")
        except Exception as e:
            print(f"   ❌ Error transcribiendo {file.name}: {e}")

    print(f"\n🎥 Audios transcritos: {len(data)}")
    return pd.DataFrame(data)

# ==============================
# EJECUCIÓN
# ==============================
run_audio_download()
df_audio = transcribe_youtube()



🚀 INICIANDO DESCARGA DE AUDIO...
   🎥 Descargando: ideas_snacks...
      ✅ Descarga OK.         
   🎥 Descargando: podcast_recomendaciones...
      ✅ Descarga OK.         
   🎥 Descargando: empresa_snacks...
      ✅ Descarga OK.         
   🎥 Descargando: realfooding_snacks...
      ✅ Descarga OK.         
   🎥 Descargando: no_experto_snacks...
      ✅ Descarga OK.         

🧠 Cargando modelo Whisper (base)...
🎧 Transcribiendo empresa_snacks.mp3
   ✅ Transcripción completa (guardada en empresa_snacks.txt)
🎧 Transcribiendo ideas_snacks.mp3
   ✅ Transcripción completa (guardada en ideas_snacks.txt)
🎧 Transcribiendo no_experto_snacks.mp3
   ✅ Transcripción completa (guardada en no_experto_snacks.txt)
🎧 Transcribiendo podcast_recomendaciones.mp3
   ✅ Transcripción completa (guardada en podcast_recomendaciones.txt)
🎧 Transcribiendo realfooding_snacks.mp3
   ✅ Transcripción completa (guardada en realfooding_snacks.txt)

🎥 Audios transcritos: 5


## 1.4. Unificación del dataset

Se combinan los DataFrames de artículos web y transcripciones de audio en un único dataset multimodal. 
Este dataset se guardará como CSV para su posterior preprocesamiento.

In [ ]:
# ==============================
# CARGAR TRANSCRIPCIONES EXISTENTES A UN DF
# ==============================
def load_audio_transcripts():
    data = []
    for file in TRANSCRIPTS_PATH.glob("*.txt"):
        try:
            content = file.read_text(encoding="utf-8", errors="ignore")
            
            # Extraer título (nombre de archivo)
            title = file.stem
            
            # Limpiar texto para content: quitar encabezados si los hay
            lines = content.splitlines()
            text_lines = [line for line in lines if not line.startswith("TITULO:") and not line.startswith("SOURCE:") and not line.startswith("-"*5)]
            text_content = " ".join(text_lines).strip()
            
            data.append({
                "source": "youtube",
                "title": title,
                "type": "audio",
                "content": text_content
            })
        except Exception as e:
            print(f"⚠️ Error leyendo {file.name}: {e}")
    
    print(f"\n🎥 Transcripciones cargadas: {len(data)}")
    return pd.DataFrame(data)

# ==============================
# EJECUCIÓN
# ==============================
df_audio = load_audio_transcripts()


🎥 Transcripciones cargadas: 5


In [ ]:
# ==============================
# UNIR DF WEB Y TRANSCRIPCIONES VÍDEOS YOUTUBE
# ==============================
df_final = pd.concat([df_web, df_audio], ignore_index=True)

# ==============================
# GUARDAR CSV
# ==============================
output_csv = OUTPUT_PATH / f"01_dataset_multimodal.csv"

with open(output_csv, "w", encoding="utf-8", newline="") as f:
    df_final.to_csv(f, index=False, quoting=csv.QUOTE_ALL)

print("\n✅ DATASET FINAL GUARDADO EN:")
print(output_csv)
print(f"📊 TOTAL FILAS: {len(df_final)}")


✅ DATASET FINAL GUARDADO EN:
h:\Cristina\Mis documentos\Github\TFM-2-A.Multimodal_snacks_saludables\data\processed\01_dataset_multimodal.csv
📊 TOTAL FILAS: 17


## Limpieza del texto

Se limpian los textos eliminando saltos de línea, URLs, caracteres especiales y se eliminan duplicados y textos muy cortos. 
Se genera un dataset minimal, optimizado para análisis con modelos LLM y generación de insights.

In [ ]:
# ==============================
# CARGAR EL CSV
# ==============================
df = pd.read_csv(output_csv)

# ==============================
# LIMPIEZA DEL TEXTO
# ==============================
# Eliminamos saltos de línea, URLs y caracteres no alfabéticos
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-záéíóúüñ0-9.,;:()¿?¡! ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['content_clean'] = df['content'].apply(clean_text)

# Eliminar duplicados y textos muy cortos
df = df.drop_duplicates(subset=['content_clean'])
df = df[df['content_clean'].str.strip() != ""]
df = df[df['content_clean'].str.split().apply(len) >= 5].reset_index(drop=True)

In [ ]:
# ==============================
# GUARDADO DEL DATASET PREPROCESADO
# ==============================
output_preprocessed = OUTPUT_PATH / "02_dataset_preprocessed.csv"
df[['content_clean', 'source', 'type']].to_csv(output_preprocessed, index=False)

print("\n✅ Dataset minimal para LLM guardado en:")
print(output_preprocessed)
print("Número de textos tras preprocesamiento:", len(df))
print("\nEjemplo del dataset minimal:")
print(df[['content_clean', 'source', 'type']].head())


✅ Dataset minimal para LLM guardado en:
h:\Cristina\Mis documentos\Github\TFM-2-A.Multimodal_snacks_saludables\data\processed\02_dataset_preprocessed.csv
Número de textos tras preprocesamiento: 17

Ejemplo del dataset minimal:
                                       content_clean            source type
0  titulo: recetas saludables contra el frío inve...  directoalpaladar  web
1  titulo: ejercicio físico en verano url: practi...              nutt  web
2  titulo: el agua en las comidas no engorda url:...              nutt  web
3  titulo: productos bio, veganos, sin gluten, si...        solnatural  web
4  titulo: calabacín rebozado y crujiente en frei...          vitonica  web
